# Exploratory Data Analysis – UNSW‑NB15 Dataset

This notebook performs exploratory data analysis on the UNSW‑NB15 dataset to
understand its structure, feature composition, class distribution, and suitability
for binary alert triage in a Security Operations Center (SOC) context.

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from pathlib import Path

DATA_DIR = Path("../data/UNSW-NB15 Dataset/UNSW-NB15 dataset/CSV Files/Training and Testing Sets")


Dataset loading

In [2]:

train = pd.read_csv(DATA_DIR / "UNSW_NB15_training-set.csv")
test  = pd.read_csv(DATA_DIR / "UNSW_NB15_testing-set.csv")

print(train.shape)
print(test.shape)

(175341, 45)
(82332, 45)


In [3]:
train.head()
print("=========================================")
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 175341 entries, 0 to 175340
Data columns (total 45 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 175341 non-null  int64  
 1   dur                175341 non-null  float64
 2   proto              175341 non-null  object 
 3   service            175341 non-null  object 
 4   state              175341 non-null  object 
 5   spkts              175341 non-null  int64  
 6   dpkts              175341 non-null  int64  
 7   sbytes             175341 non-null  int64  
 8   dbytes             175341 non-null  int64  
 9   rate               175341 non-null  float64
 10  sttl               175341 non-null  int64  
 11  dttl               175341 non-null  int64  
 12  sload              175341 non-null  float64
 13  dload              175341 non-null  float64
 14  sloss              175341 non-null  int64  
 15  dloss              175341 non-null  int64  
 16  si

In [4]:
train[['label', 'attack_cat']].value_counts()

label  attack_cat    
0      Normal            56000
1      Generic           40000
       Exploits          33393
       Fuzzers           18184
       DoS               12264
       Reconnaissance    10491
       Analysis           2000
       Backdoor           1746
       Shellcode          1133
       Worms               130
Name: count, dtype: int64

In [5]:
train['label'].value_counts(normalize=True)

label
1    0.680622
0    0.319378
Name: proportion, dtype: float64

In [ ]:
train.select_dtypes(include='object').columns

Index(['proto', 'service', 'state', 'attack_cat'], dtype='object')

: 

Interpreting the above results:

Dataset size 
Training: 82,332 rows
Testing: 175,341 rows

Feature composition 
45 total columns
44 features + 1 label
No missing values
Mix of:
30 integer
11 float
4 categorical This is a good scenario for XGBoost.
Target variable clarity (critical) We have two labels, but only one true target:
label → binary target
0 = Normal
1 = Attack
attack_cat → metadata
Attack type (Generic, Exploits, DoS, etc.)
This aligns perfectly with our Aim: binary classification for alert triage

Class balance (this is subtle but important)
Attacks: 55.06%
Normal: 44.94% This is not heavily imbalanced.
That means:

Accuracy is not totally useless
But precision & recall still matter (SOC reality)
Categorical features (future pain, but manageable)
Identified exactly what was expected: proto service state attack_cat

Immediate insight:

attack_cat cannot be used as a feature

It leaks ground truth
The other three:

Need encoding later
Are classic network-protocol fields
Add real SOC context

We now understand:
Shape
Features
Labels
Balance
Data types
Hidden traps

Feature Selection Decisions
Columns that must be removed There is one obvious one:  id

Pure identifier
No predictive meaning
Dangerous noise
Columns that must be excluded from features : attack_cat

Encodes the type of attack
Available only because you already know it’s an attack
Using it would be academic fraud (even accidentally)

We keep it:
For analysis
For reporting
Not for training
Columns that are candidates to keep

Everything else, for now.

We don’t prune aggressively yet. XGBoost can handle noise.

### Summary

The exploratory analysis confirms that the UNSW‑NB15 dataset is well suited for
binary alert triage tasks. The dataset exhibits a balanced class distribution,
no missing values, and a mix of numerical and categorical features, making it
appropriate for tree‑based baseline models such as XGBoost.
